# Notebook 4: Search Result Evaluation
How do we know if the results of a search are on topic? 
The goal: Be able to evaluate topic search results. 

In [1]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

# Outline: 

- What is evaluation-K? Why do we set that? (search depth)
- What is precision@K
- What is recall@K? 
- What do we mean by "baseline precision" and how do you calculate it? 

## Succes Metric

> **"Ratio of posts assigned to a topic that come back in topic search."**

If we run a topic-embedding search and the top results are *not* the posts that
BERTopic itself put in that topic, the topic embedding is a poor query — and the
user-facing search will surface unrelated posts.

We measure this with three numbers per topic:

1. **Precision@K** — of the top-K retrieved posts, how many were actually assigned
   to this topic? (Tracks the *display* experience — what the user sees first.)
2. **Recall@K** — of the posts assigned to this topic, how many appeared in the top-K?
   (Tracks *coverage*.)
3. **Random baseline precision** — what precision would we expect from random ranking?
   (Tells us how much our search is doing above chance.)

## Three different K values — keep them straight

(Definitions reproduced from `src/evaluation.py`.)

- **eval-K (retrieval depth)** — how many results we *retrieve* per topic for
  evaluation. This is the `size` parameter on the search query. Set it large enough
  to plausibly cover all of a topic's posts; small enough that we're not dragging in
  half the corpus. This is not included in the metrics and just used to set the
  search depth. 
- **recall-K (evaluation cutoff)** — the cutoff used in the recall calculation. Often
  equal to eval-K.
- **precision-K (display cutoff)** — typically a small number like 8 or 10 — the number
  of results a user actually sees on the first page.

Defaults in this repo: `DEFAULT_EVAL_K=100`, `DEFAULT_RECALL_K=100`, `DEFAULT_PRECISION_K=8`.

**Random baseline precision** for a topic of size *t* in a corpus of size *N* is just
`t / N` — the chance that a randomly-picked post happens to be in the topic. 

## How to compute the three metrics

Imagine a topic with 3 ground-truth posts: `{post-a, post-b, post-c}`, in a corpus of 10 posts. A
search returned 5 ranked results: `[post-a, post-b, post-c, post-d, post-e]`. 

We'll compute precision@3,
recall@3, and the random baseline by hand.

In [2]:
retrieved_ids = ["post-a", "post-b", "post-c", "post-d", "post-e"]    # search results, ranked
topic_post_ids = {"post-a", "post-c", "post-x"}             # ground truth
k = 3
dataset_size = 10

# --- Precision@K ---
# Of the top-K retrieved, how many are in the ground-truth set?
top_k = retrieved_ids[:k]                    # ['post-a', 'post-b', 'post-c']
hits = sum(pid in topic_post_ids for pid in top_k)   # post-a✓ post-b✗ post-c✓ → 2
precision_at_k = hits / len(top_k)           # 2 / 3 = 0.667
print(f"precision@{k} = {hits} hits / {len(top_k)} retrieved = {precision_at_k:.3f}")

# --- Recall@K ---
# Of the ground-truth set, how many showed up in the top-K retrieved?
recall_at_k = hits / len(topic_post_ids)     # 2 / 3 = 0.667
print(f"recall@{k}    = {hits} hits / {len(topic_post_ids)} ground-truth = {recall_at_k:.3f}")

# --- Random baseline precision ---
# If we picked posts uniformly at random, what fraction would be on-topic?
baseline = len(topic_post_ids) / dataset_size  # 3 / 10 = 0.300
print(f"baseline     = {len(topic_post_ids)} / {dataset_size} = {baseline:.3f}")
print(f"\nLift over random: {precision_at_k - baseline:+.3f}")

precision@3 = 2 hits / 3 retrieved = 0.667
recall@3    = 2 hits / 3 ground-truth = 0.667
baseline     = 3 / 10 = 0.300

Lift over random: +0.367


## Exercise

Time: 5 minutes

Given what you learned above fill in three function bodies in `src/evaluation.py`.  

Edge cases worth noting:
- Search returned fewer than K results
- Topic has zero ground-truth posts  `recall@K` would divide by zero  
- Empty dataset -- same, guard against dividing by zero.

## Interpreting the numbers

Once your `src/evaluation.py` functions return real numbers, open the **Topic
Evaluation** toggle in the demo app (it's wired up in `app.py` already). For each
topic you'll see `precision@K`, `recall@K`, and the random baseline side-by-side.

Things to look for:
- Topics that beat baseline by 50× or more → search is doing real work.
- Topics that barely beat baseline → either the topic is small or the topic
  embedding is diffuse (we'll fix that in Notebook 5).
- Topics where recall@K is high but precision@K is low → the topic is well-covered
  but the *first* results aren't the best ones, often because of close-by topics.

# Exercise: 

1. Based on what you learned in this notebook, update the placeholder code in this
file to compute baseline precision, precision@k and recall@k. 

```
src/evaluation.py 
```

2. Then open the demo
app and evaluate the results by toggling "Topic Evaluation" How do different
topics compare? Which topics perform well? Which don't? Why? 

See reference implementations in `solutions/evaluation.py`.